# Brazil Data Cube — CBERS-4 / AMAZONIA-1 / MODIS VI

The `bdc` endpoint (INPE's Brazil Data Cube STAC v1) is the **only global source** of CBERS-4 /
CBERS-4A and AMAZONIA-1 imagery, plus 16-day MODIS Terra/Aqua vegetation indices over South
America. Every currently-exposed BDC collection reads **anonymously** (verified by the A1 token
probe).

Forward-compat note: earthlens ships a `BdcTokenSigner` + `Collection.requires_token` infra so a
future BDC tier requiring `$BDC_ACCESS_TOKEN` can land catalog-only. Today no curated row uses
either.

## The endpoint + curated collections (no network)

In [ ]:
from earthlens.stac import Catalog

cat = Catalog()
e = cat.endpoints["bdc"]
print(f"url   : {e.url}")
print(f"signer: {e.signer}")
bdc_keys = [k for k in cat.datasets if k.startswith("bdc/")]
print(f"curated: {bdc_keys}")

## CBERS-4 WFI 16-day NDVI composite over São Paulo (live)

`CBERS4-WFI-16D-2` is BDC's flagship cube — every 16 days, every pixel south of Mexico, fused
from the China-Brazil CBERS-4 WFI sensor. The `NDVI` asset is a single-band float COG.

In [ ]:
import tempfile

from earthlens.core import EarthLens

el = EarthLens(
    data_source="bdc",  # alias 'brazil-data-cube' works too
    start="2024-01-01",
    end="2024-12-31",
    dataset="bdc/CBERS4-WFI-16D-2",
    variables=["NDVI"],
    aoi=[-46.8, -23.7, -46.3, -23.2],  # São Paulo
    path=tempfile.mkdtemp(),
    max_items=1,
)
paths = el.download()
paths

## AMAZONIA-1 WFI surface reflectance (recipe)

Same request shape — only `dataset` and `variables` change. AMAZONIA-1 is Brazil's first
domestically-built Earth-observation satellite; BDC is the only public source globally.

In [ ]:
amz = EarthLens(
    data_source="bdc",
    start="2024-01-01",
    end="2024-12-31",
    dataset="bdc/AMZ1-WFI-L4-SR-1",
    variables=["BAND2", "BAND1", "BAND3"],  # red, green, NIR
    aoi=[-60.0, -3.5, -59.5, -3.0],  # over the Amazon basin
    path=tempfile.mkdtemp(),
    max_items=1,
)
amz_paths = amz.download()
amz_paths

## Inspect

In [ ]:
from pyramids.dataset import Dataset

for label, ps in (("CBERS-4 NDVI", paths), ("AMAZONIA-1 SR", amz_paths)):
    ds = Dataset.read_file(str(ps[0]))
    print(
        f"{label:18} bands={ds.band_count} epsg={ds.epsg} shape={ds.read_array().shape}"
    )

## See also

- Migration mapping for `MODIS/061/MOD13Q1` / `MYD13Q1` → `bdc/mod13q1-6.1` / `myd13q1-6.1` in
  `docs/reference/stac/migration-from-gee.md`.
- The `BdcTokenSigner` recipe lives in `docs/reference/stac/authentication.md` — wire it in via a
  per-collection `signer: bdc-token` override when INPE next exposes a token-gated tier.